## Chargement des Données via SQLAlchemy

Configurer la connexion SQLAlchemy

In [ ]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")

if not all([user, password, db]):
    raise ValueError("One or more DB environment variables are missing! Check your .env file.")

# connection string
engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{db}")

# test connection
try:
    with engine.connect() as conn:
        print('Connection successful!')
except Exception as e:
    print(f'connection failed: {e}')

Séparer financecore_clean.csv selon les tables normalisées avant insertion


In [ ]:
import pandas as pd
df = pd.read_csv("../data/financecore_clean.csv")

print(df.head())
print(df.columns)

In [ ]:
# Extract unique Agencies
agencies_df = df[['agence']].drop_duplicates().reset_index(drop=True)

# Extract unique Products
products_df = df[['produit']].drop_duplicates().reset_index(drop=True)

# Extract unique Categories
categories_df = df[['categorie']].drop_duplicates().reset_index(drop=True)

# Extract unique Clients (with their associated metadata)
# We drop duplicates based on 'client_id' to ensure 1
#  row per client
clients_df = df[['client_id', 'score_credit_client', 'categorie_risque', 'segment_client']].drop_duplicates(subset=['client_id'])

# IMPORTANT: Rename 'categorie_risque' to match your SQL 'categories_risque' (with the 's')
clients_df = clients_df.rename(columns={'categorie_risque': 'categories_risque'})

#checkpoint: print the count of unique ites found
print(f'extraction complete:')
print(f'- {len(agencies_df)} Unique agencies')
print(f'- {len(products_df)} Unique products')
print(f'- {len(categories_df)} Unique categories')
print('f- {len(clients_df)} Unique')